# 🛡️ Glu-Stock: 01_RESEARCH_SCAN
**Phase**: Universe Selection & Fundamental Filtering

This notebook scans the IDX market, filters for liquidity and high-growth fundamentals, and pushes candidates to the Firebase `research` queue.

In [ ]:
# 📦 SECTION 1: INSTALLATION
!pip install -q yfinance firebase-admin pandas scikit-learn joblib python-dotenv


In [ ]:

def get_dynamic_lq45():
    print("🌐 Fetching latest LQ45 constituents...")
    fallback = ["ACES.JK", "ADRO.JK", "AKRA.JK", "AMMN.JK", "AMRT.JK", "ANTM.JK", "ARTO.JK", "ASII.JK", "BBCA.JK", "BBNI.JK", "BBRI.JK", "BBTN.JK", "BMRI.JK", "BRIS.JK", "BRPT.JK", "BUKA.JK", "CPIN.JK", "CTRA.JK", "ESSA.JK", "EXCL.JK", "GGRM.JK", "GOTO.JK", "HRUM.JK", "ICBP.JK", "INCO.JK", "INDF.JK", "INKP.JK", "INTP.JK", "ISAT.JK", "ITMG.JK", "KLBF.JK", "MAPI.JK", "MBMA.JK", "MDKA.JK", "MEDC.JK", "MTEL.JK", "PGAS.JK", "PGEO.JK", "PTBA.JK", "SIDO.JK", "SMGR.JK", "SRTG.JK", "TLKM.JK", "TPIA.JK", "UNTR.JK"]
    try:
        import urllib.request
        req = urllib.request.Request('https://raw.githubusercontent.com/yofriadi/idn-stock-list/master/lq45.json', headers={'User-Agent': 'Mozilla/5.0'})
        with urllib.request.urlopen(req, timeout=5) as url:
            data = json.loads(url.read().decode())
            return [f"{t}.JK" for t in data]
    except:
        print("⚠️ External fetch failed. Using highly-curated fallback LQ45 list.")
    return fallback

def get_full_idx_universe():
    print("🌐 Fetching ALL IDX listed companies from Official IDX API...")
    fallback = get_dynamic_lq45() # Fallback to LQ45 if fail
    
    # 1. Try Official IDX API (Hit Network)
    try:
        import urllib.request
        # Appending length=9999 handles pagination if the API enforces it
        req = urllib.request.Request('https://www.idx.co.id/primary/StockData/GetSecuritiesStock?length=9999', headers={'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64)'})
        with urllib.request.urlopen(req, timeout=10) as url:
            data = json.loads(url.read().decode())
            if 'data' in data:
                tickers = [f"{t['Code']}.JK" for t in data['data'] if 'Code' in t]
                if tickers:
                    print(f"✅ Successfully fetched {len(tickers)} companies from IDX Official API.")
                    return list(set(tickers)) # Unique check
    except Exception as e:
        print(f"⚠️ Official IDX API failed. Trying Github Proxy...")
        
    # 2. Try Github Alternative
    try:
        import urllib.request
        req = urllib.request.Request('https://raw.githubusercontent.com/yofriadi/idn-stock-list/master/stock-list.json', headers={'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64)'})
        with urllib.request.urlopen(req, timeout=10) as url:
            data = json.loads(url.read().decode())
            tickers = [f"{t['ticker']}.JK" for t in data if 'ticker' in t]
            if tickers:
                print(f"✅ Successfully fetched {len(tickers)} companies from Github proxy.")
                return tickers
    except Exception as e:
        print("⚠️ Full fetch failed. Falling back to LQ45.")
        
    return fallback


In [ ]:
# 🧠 SECTION 3: CORE LOGIC (Research Agent)
import yfinance as yf
import pandas as pd

class ResearchAgent:
    def get_market_data(self, ticker: str, period: str = "1y") -> pd.DataFrame:
        data = yf.download(ticker, period=period, interval="1d", progress=False)
        return data

    def fundamental_filter(self, ticker: str) -> bool:
        try:
            info = yf.Ticker(ticker).info
            # Filter logic: Growth + Valuation
            growth = info.get('earningsQuarterlyGrowth', 0) or 0
            pe = info.get('trailingPE', 100)
            return growth > 0.05 and pe < 25
        except: return False

In [ ]:
# 🚀 SECTION 4: MAIN EXECUTION
def run_scan():
    secrets = KaggleInfra.load_secrets()
    fb = FirebaseHandler(secrets)
    research = ResearchAgent()
    
    universe = get_full_idx_universe()
    candidates = []
    
    print(f"🔭 Scanning {len(universe)} stocks...")
    for ticker in universe:
        if research.fundamental_filter(ticker):
            candidates.append(ticker)
            print(f"✅ {ticker} passed fundamental filter.")
            
    if candidates:
        fb.push_task("research", candidates)
        fb.log_event("RESEARCH", f"Pushed {len(candidates)} stocks to queue.")
        print(f"🚀 Successfully pushed analysis tasks for {candidates}")
    else:
        print("💤 No opportunities found today.")

run_scan()